In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "src").exists() and (p / "notebooks").exists():
            return p
    return start


ROOT = find_repo_root(Path.cwd())
os.chdir(ROOT)

FREQ_PATH = ROOT / "data" / "staging" / "freq_staged.parquet"

freq = pd.read_parquet(FREQ_PATH)
freq.shape, freq.columns.tolist()

((678013, 12),
 ['IDpol',
  'ClaimNb',
  'Exposure',
  'Area',
  'VehPower',
  'VehAge',
  'DrivAge',
  'BonusMalus',
  'VehBrand',
  'VehGas',
  'Density',
  'Region'])

In [ ]:
s = freq["ClaimNb"]
exp = freq["Exposure"]

summary = {
    "rows": int(len(freq)),
    "claimnb_mean_per_row": float(s.mean()),
    "claimnb_var_per_row": float(s.var()),
    "zero_rate": float((s == 0).mean()),
    "max": int(s.max()),
    "p95": float(s.quantile(0.95)),
    "p99": float(s.quantile(0.99)),
    # exposure-weighted overall claim frequency (claims per exposure-year)
    "lambda_hat_total_claims_over_total_exposure": float(s.sum() / exp.sum()),
    "total_exposure": float(exp.sum()),
}
summary

{'rows': 678013,
 'mean': 0.05324676665491664,
 'var': 0.057656332380878514,
 'zero_rate': 0.9497649750078538,
 'max': 16,
 'p95': 1.0,
 'p99': 1.0}

In [ ]:
exp = freq["Exposure"]
annualized = freq["ClaimNb"] / exp

# Annualized rates can be dominated by tiny exposures.
# Report both: (a) full-population stats and (b) a more stable view excluding very small exposure rows.
exp_floor = 0.05
mask = exp >= exp_floor

profile = {
    "exposure_min": float(exp.min()),
    "exposure_p01": float(exp.quantile(0.01)),
    "exposure_median": float(exp.quantile(0.50)),
    "exposure_max": float(exp.max()),
    "rows_exposure_lt_floor": int((~mask).sum()),
    "rows_exposure_ge_floor": int(mask.sum()),
    "annualized_mean_all": float(annualized.mean()),
    "annualized_p95_all": float(annualized.quantile(0.95)),
    "annualized_p99_all": float(annualized.quantile(0.99)),
    "annualized_mean_ge_floor": float(annualized[mask].mean()),
    "annualized_p95_ge_floor": float(annualized[mask].quantile(0.95)),
    "annualized_p99_ge_floor": float(annualized[mask].quantile(0.99)),
}
profile

{'exposure_min': 0.00273224,
 'exposure_p01': 0.0082191780821917,
 'exposure_median': 0.49,
 'exposure_max': 1.0,
 'annualized_mean': 0.26397243991310276,
 'annualized_p95': 1.0,
 'annualized_p99': 4.166666666666667}

In [ ]:
y = freq["ClaimNb"].to_numpy()
exp = freq["Exposure"].to_numpy()

# Baseline intercept-only Poisson with offset (no features):
# mu_i = Exposure_i * lambda_hat, where lambda_hat = total_claims / total_exposure
lambda_hat = float(y.sum() / exp.sum())
mu = exp * lambda_hat

observed_zero_rate = float((y == 0).mean())

# Poisson-implied *marginal* zero rate under varying mu_i is E[exp(-mu_i)]
expected_zero_rate = float(np.mean(np.exp(-mu)))

# Simple dispersion diagnostic (Pearson): sum((y-mu)^2/mu) / (n-1)
# (n-1 as a proxy for df after estimating 1 parameter: lambda_hat)
with np.errstate(divide="ignore", invalid="ignore"):
    pearson_terms = (y - mu) ** 2 / mu
pearson_terms = pearson_terms[np.isfinite(pearson_terms)]
dispersion = float(pearson_terms.sum() / max(1, (len(mu) - 1)))

{
    "lambda_hat_total_claims_over_total_exposure": lambda_hat,
    "observed_zero_rate": observed_zero_rate,
    "expected_zero_rate_poisson_mixture": expected_zero_rate,
    "zero_rate_gap_obs_minus_exp": float(observed_zero_rate - expected_zero_rate),
    "pearson_dispersion_intercept_only": dispersion,
}

{'mean': 0.05324676665491664,
 'var': 0.057656332380878514,
 'var_over_mean': 1.0828137744877455,
 'zero_rate': 0.9497649750078538}

### Baseline Poisson check (with Exposure offset)

Because `Exposure` varies across rows, `ClaimNb` marginally behaves like a **mixture of Poissons** even if the conditional model is Poisson. So unconditional checks like `var(ClaimNb)/mean(ClaimNb)` can be misleading.

Instead, we do an **intercept-only Poisson with offset**:

- Estimate \(\hat\lambda = \sum ClaimNb / \sum Exposure\)
- Compute \(\mu_i = Exposure_i \times \hat\lambda\)
- Compare observed zero rate vs \(E[\exp(-\mu_i)]\)
- Compute a simple Pearson dispersion diagnostic (values \(\gg 1\) suggest overdispersion beyond the offset-only Poisson)

**Interpretation:**
- If dispersion is close to 1, Poisson is a reasonable baseline.
- If dispersion is materially above 1, consider NB (or add features / random effects) before deciding Poisson is “adequate”.

In [ ]:
freq["ClaimNb"].value_counts().sort_index().plot(kind="bar")

In [ ]:
# Quick segment-level sanity check: exposure-weighted claim rate by category
# (This is descriptive, not a model.)

def rate_by(col: str, top_n: int = 10):
    if col not in freq.columns:
        return None
    g = (
        freq.groupby(col, dropna=False)
        .agg(claims=("ClaimNb", "sum"), exposure=("Exposure", "sum"), rows=("IDpol", "count"))
        .assign(rate=lambda d: d["claims"] / d["exposure"])
        .sort_values("exposure", ascending=False)
    )
    return g.head(top_n)

{
    "rate_by_region_top10_by_exposure": rate_by("Region", top_n=10),
    "rate_by_vehgas": rate_by("VehGas", top_n=10),
}

### Conclusion

- **Exposure varies materially**, including a non-trivial mass of very small exposures. Any per-policy “annualized” rate (`ClaimNb/Exposure`) should be interpreted with care and typically summarized with an exposure floor or exposure-weighting.
- The most useful baseline frequency for pricing is the **exposure-weighted rate** \(\hat\lambda = \sum ClaimNb / \sum Exposure\) (claims per exposure-year).
- A more appropriate sanity check than unconditional variance/mean is the **offset-aware intercept-only Poisson** comparison (observed zero rate vs \(E[e^{-\mu_i}]\)) plus a **dispersion diagnostic**.

### Recommended next steps
- Fit a **Poisson GLM with `log(Exposure)` offset** using core rating factors (`Area`, `Region`, `BonusMalus`, `VehAge`, `VehPower`, `DrivAge`, `VehBrand`, `VehGas`, `Density`).
- If the dispersion diagnostic remains **materially above 1** after adding features, consider **Negative Binomial** (or additional structure such as random effects / credibility).
- Produce **decile calibration** (Observed vs Expected claims per exposure) and segment stability checks before locking a frequency model for pricing.